<a href="https://colab.research.google.com/github/jaroslav200720072007-beep/fashion-mnist-classifier/blob/main/Image_Classification_using_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [107]:
import torch

In [108]:
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda, Compose
import matplotlib.pyplot as plt

In [109]:
##Download data
training_data = datasets.FashionMNIST(
    root = "data",
    train = True,
    download = True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform=ToTensor()
)

In [110]:
batch_size = 64

train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

In [111]:
for x, y in test_dataloader:
  print(x, y)

Выходные данные были обрезаны до нескольких последних строк (5000).
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]]],


        [[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]]],


        ...,


        [[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.

In [112]:
device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available() else 'cpu'
)
print(f'Використовується пристрій: {device}')

Використовується пристрій: cpu


In [113]:
## Define our custom Model

class NeuralNetworks(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear_relu_stack = nn.Sequential(
        nn.Flatten(),
        nn.Linear(28*28, 512),
        nn.ReLU(),
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512, 10)
    )

  def forward(self, x):
    return self.linear_relu_stack(x)

In [114]:
device

device(type='cpu')

In [115]:
model = NeuralNetworks().to(device)
print(model)

NeuralNetworks(
  (linear_relu_stack): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=512, bias=True)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=512, bias=True)
    (4): ReLU()
    (5): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [116]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 1e-3)

In [117]:
##Train loop

def train(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset)
  model.train()

  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)

    ##Calculate prediction error
    pred = model(X)
    loss = loss_fn(pred, y)

    ## Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if batch % 100 == 0:
      loss, current = loss.item(), batch * len(X)
      print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

In [118]:
def test(dataloader, model, loss_fn):
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  model.eval()
  test_loss, correct = 0, 0
  with torch.no_grad():
    for X, y in dataloader:
      X, y = X.to(device), y.to(device)
      pred = model(X)
      test_loss += loss_fn(pred, y).item()
      correct += (pred.argmax(1) == y).type(torch.float).sum().item()
  test_loss /= num_batches
  correct /= size
  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f}")

In [120]:
## Start training

epochs = 50

for t in range(epochs):
  train(train_dataloader, model, loss_fn, optimizer)
  if (t + 1) % 10 == 0:
    print(f"Epoch {t+1}-------------------------")
    test(test_dataloader, model, loss_fn)

print("Done!")

loss: 1.373781 [    0/60000]
loss: 1.356286 [ 6400/60000]
loss: 1.184660 [12800/60000]
loss: 1.294238 [19200/60000]
loss: 1.167220 [25600/60000]
loss: 1.179655 [32000/60000]
loss: 1.218076 [38400/60000]
loss: 1.142710 [44800/60000]
loss: 1.182760 [51200/60000]
loss: 1.105378 [57600/60000]
loss: 1.186140 [    0/60000]
loss: 1.190736 [ 6400/60000]
loss: 1.002727 [12800/60000]
loss: 1.141093 [19200/60000]
loss: 1.007905 [25600/60000]
loss: 1.033680 [32000/60000]
loss: 1.086509 [38400/60000]
loss: 1.016564 [44800/60000]
loss: 1.056764 [51200/60000]
loss: 0.995626 [57600/60000]
loss: 1.058625 [    0/60000]
loss: 1.084285 [ 6400/60000]
loss: 0.878310 [12800/60000]
loss: 1.040386 [19200/60000]
loss: 0.906933 [25600/60000]
loss: 0.931854 [32000/60000]
loss: 1.001143 [38400/60000]
loss: 0.934479 [44800/60000]
loss: 0.969584 [51200/60000]
loss: 0.923974 [57600/60000]
loss: 0.965538 [    0/60000]
loss: 1.011482 [ 6400/60000]
loss: 0.789567 [12800/60000]
loss: 0.970503 [19200/60000]
loss: 0.840048

In [ ]:
##Save Model

torch.save(model.state_dict(), "model.pth")
print("Model saved!")

In [ ]:
## Load Model

model = NeuralNetworks()
model.load_state_dict(torch.load("/content/model.pth"))

In [ ]:
## Prediction Classes:

classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

In [ ]:
len(classes)

In [ ]:
model.eval()

In [ ]:
x, y = test_data[0][0], test_data[0][1]
x, y

In [ ]:
with torch.no_grad():
  pred = model(x)
  predicted, actual = classes[pred[0].argmax(0)], classes[y]
  print(f'Predicted: "{predicted}", Actual: "{actual}"')